# 🚀 Final Project: Critical Evaluation of Deep Belief Networks for Intrusion Detection

## Project Overview
This notebook presents a rigorous, empirical data-science reproduction and critical evaluation of the paper:
**"An Intrusion Detection System based on Deep Belief Networks"** by Belarbi et al. (SciSec 2022).

**Repository:** `othmbela/dbn-based-nids`  
**Dataset:** CICIDS2017 (15 classes: 14 attacks + Benign)

### Central Objective
The original authors claim that Deep Representation Learning (specifically, stacked Restricted Boltzmann Machines) is fundamentally superior for Network Intrusion Detection. We critically test this claim by:
1. Re-implementing their Multi-Class architecture on the authentic dataset.
2. Expanding the evaluation protocol to include stringent Train/Val/Test isolation, Chronological Testing, and Duplicate Handling.
3. Benchmarking the DBN against classical tree-ensembles.

---


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import load_real_cicids2017, run_eda, run_duplicate_audit, clean_data, split_and_scale
from model_utils import train_all_models, evaluate_all_models, get_multiclass_error_analysis

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.figsize': (10, 5), 'figure.dpi': 120})


---
## 1. 📥 Dataset Ingestion

We load a 10% stratified sample of the massive CICIDS2017 raw CSV files. This preserves the exact class imbalance of the 2.8 million original flows while making the computational pipeline tractable for end-to-end evaluation.


In [ ]:
print("Loading real CICIDS2017 dataset...")
df = load_real_cicids2017(sample_frac=0.10, random_state=42)
print(f"Dataset shape: {df.shape}")
df.head(3)


---
## 2. 📊 Exploratory Data Analysis (EDA)

Before modeling, we conduct a deep analysis of the 78 features generated by CICFlowMeter. 
Network traffic extractors often generate pathological data, and we must identify NaNs, Infinities, and constant fields.


In [ ]:
eda_results = run_eda(df)

print(f"Missing Values (NaNs): {eda_results['missing_values']}")
print(f"Infinite Values (Infs): {eda_results['infinite_values']}")


---
## 3. 🛡️ Duplicate & Leakage Audit

Before making any splits, we conduct a massive data integrity audit. Network traffic datasets like CICIDS2017 are infamous for containing identical flows (e.g., automated port scans generate thousands of identical TCP syn packets). 

If we split randomly without deduplication, these identical flows leak into the Test set, granting models a "free pass" to cheat. We also analyze identical feature vectors with *conflicting* labels (a fundamental flaw in the dataset).


In [ ]:
audit = run_duplicate_audit(df)

print(f"Total Exact Duplicate Rows: {audit['exact_duplicates']}")
print(f"Total Duplicate Feature Vectors (ignoring label): {audit['feature_duplicates_ignoring_label']}")
print(f"Identical Feature Vectors with Conflicting Labels: {audit['conflicting_label_vectors']}")

print("\nTop 5 Days by Duplicate Volume:")
for day, count in sorted(audit['duplicates_per_day'].items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {day}: {count} duplicates")


---
## 4. 🧹 Data Cleaning & Preprocessing

The authentic codebase performs several critical preprocessing steps, which we have reproduced and expanded to prevent data leakage.


In [ ]:
df_clean, constant_cols, cols_to_drop, label_encoder = clean_data(df)
print(f"Dropped constant/zero-variance columns ({len(constant_cols)}):")
for col in constant_cols:
    print(f"  ✗ {col}")
    
# Many column names in CICIDS2017 are identical but with trailing spaces.
dropped_duplicate = [c for c in cols_to_drop if c not in constant_cols]
print(f"\nDropped duplicate columns ({len(dropped_duplicate)}):")
for col in dropped_duplicate:
    print(f"  ✗ {col}")
print(f"\nRemaining features: {df_clean.shape[1] - 2}")  # minus Label and Day


---
## 5. ✂️ Experimental Split Protocols (Leakage Analysis)

To measure the true impact of duplicate data leakage, we evaluate **four** distinct splitting strategies using strict 60/20/20 Train/Val/Test isolation:

1. **Random Original**: The baseline 60/20/20 stratified split on the raw data (contains duplicates).
2. **Random Deduplicated**: We explicitly drop exact duplicates from the dataset *before* performing the random split.
3. **Grouped Split**: We hash the feature vectors and use a GroupShuffleSplit. This guarantees that all identical flows are isolated completely to either the Train set or the Test set.
4. **Chronological**: Train on Mon-Wed, Validate on Thu, Test on Fri.

*Data Leakage Measurement*: For each split, we calculate the percentage of Test records whose feature vectors already exist in the Training set.


In [ ]:
import json
with open('../results/figures/leakage_analysis.json', 'r') as f:
    leakage_stats = json.load(f)

print("Test Leakage Per Strategy:")
for stat in leakage_stats:
    print(f"Strategy: {stat['Strategy'].ljust(20)} | Test Leakage: {stat['Leakage_Pct']:.2f}%")


---
## 6. 🤖 Model Training & Results across Splitting Strategies

Notice the significant performance degradation across the strategies. As we eliminate data leakage (moving from Random Original -> Grouped -> Chronological), the F1-Scores plummet. The PyTorch DBN was trained specifically on `random_original` and `chronological` to satisfy the reproduction requirement while remaining computationally tractable.


In [ ]:
from IPython.display import Image, display
print("1. Random Original (Baseline with Duplicates)")
display(Image(filename='../results/figures/results_table_random_original.png'))
display(Image(filename='../results/figures/confusion_random_original.png'))

print("\n2. Random Deduplicated (First-occurrence retention prior to split)")
display(Image(filename='../results/figures/results_table_random_deduplicated.png'))
display(Image(filename='../results/figures/confusion_random_deduplicated.png'))

print("\n3. Grouped Split (0% Leakage Guarantee)")
display(Image(filename='../results/figures/results_table_grouped.png'))
display(Image(filename='../results/figures/confusion_grouped.png'))

print("\n4. Chronological Split (Realistic Generalization)")
display(Image(filename='../results/figures/results_table_chronological.png'))
display(Image(filename='../results/figures/confusion_chronological.png'))


---
## 7. 🕵️ Conclusion & Generalization Analysis

The findings definitively demonstrate that evaluating NIDS purely on random splits over dataset containing duplicates creates a false sense of security. The models successfully memorized attack signatures in the `random_original` split due to high Train-Test leakage, but struggled to adapt to novel temporal conditions and shifting distributions in the `chronological` setup. 

Furthermore, tree-based models like Random Forest and XGBoost match or exceed the performance of the Deep Belief Network at a fraction of the computational cost, refuting the paper's central claim that DBNs are strictly superior.
